# Assignment 8 - VAE on MNIST (2D latent space)

Basic VAE, 2D latent space, trained for 10 epochs on MNIST.
Tracks reconstruction loss and KL loss separately, then plots the latent space and a decoded grid.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

len(train_data), len(test_data)

## Model

In [ ]:
latent_dim = 2

class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        # encoder
        self.fc1 = nn.Linear(784, 400)
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)
        # decoder
        self.fc2 = nn.Linear(latent_dim, 400)
        self.fc3 = nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

model = VAE(latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Loss function

BCE for reconstruction + KL divergence, kept separate so we can track them individually.

In [ ]:
def loss_function(recon_x, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss, kl_loss

## Training loop

10 epochs.

In [ ]:
num_epochs = 10

recon_history = []
kl_history = []

for epoch in range(num_epochs):
    model.train()
    total_recon = 0
    total_kl = 0

    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()

        recon, mu, logvar = model(x)
        recon_loss, kl_loss = loss_function(recon, x, mu, logvar)
        loss = recon_loss + kl_loss

        loss.backward()
        optimizer.step()

        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    avg_recon = total_recon / len(train_data)
    avg_kl = total_kl / len(train_data)
    recon_history.append(avg_recon)
    kl_history.append(avg_kl)

    print(f"epoch {epoch+1}/{num_epochs}  recon_loss={avg_recon:.4f}  kl_loss={avg_kl:.4f}")

In [ ]:
plt.plot(recon_history, label="reconstruction loss")
plt.plot(kl_history, label="KL loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Training losses")
plt.show()

## Latent space scatter plot

Pass test images through the encoder and plot the 2D means, coloured by digit label.

In [ ]:
model.eval()

all_mu = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        mu, logvar = model.encode(x.view(-1, 784))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu = np.concatenate(all_mu, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(all_mu[:, 0], all_mu[:, 1], c=all_labels, cmap="tab10", s=5)
plt.colorbar(scatter, label="digit")
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Latent space (test set)")
plt.show()

## Decoded image grid

Sample a grid of points from -3 to 3 in both latent dimensions and decode each one.

In [ ]:
grid_size = 15
grid_x = np.linspace(-3, 3, grid_size)
grid_y = np.linspace(-3, 3, grid_size)

fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            img = model.decode(z).cpu().view(28, 28).numpy()
            axes[i, j].imshow(img, cmap="gray")
            axes[i, j].axis("off")

plt.suptitle("Decoded digits across the latent space")
plt.show()

## Observations

- Similar digits cluster together in the latent space, though with only 2 dimensions and 10 epochs the clusters overlap more than a higher-dimensional VAE would.
- Digits that look visually similar (e.g. 4/9, 3/5/8) sit closer together and their clusters are less clearly separated than digits with distinct shapes (e.g. 0, 1).
- Moving through the latent space in the decoded grid shows one digit gradually morphing into another rather than jumping abruptly.
- The transitions are mostly smooth, but a few regions between clusters produce blurry, ambiguous digits since those points don't correspond to any real training example.